In [3]:
!pip install -q langchain langchain-groq langchain-text-splitters langchain-huggingface langchain-community pypdf sentence-transformers faiss-cpu

In [6]:
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from google.colab import userdata
import os

# Setup Groq
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

# Setup embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("✅ LangChain ready")
print(f"🤖 Model: llama-3.3-70b-versatile")
print(f"📐 Embeddings: all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ LangChain ready
🤖 Model: llama-3.3-70b-versatile
📐 Embeddings: all-MiniLM-L6-v2


In [7]:
from google.colab import files

# Upload PDF
print("📄 Upload a PDF document...")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Load PDF — LangChain handles this in 1 line
loader = PyPDFLoader(filename)
documents = loader.load()
print(f"✅ Loaded: {len(documents)} pages")

# Split into chunks — LangChain handles this in 3 lines
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)
chunks = splitter.split_documents(documents)
print(f"✅ Split into {len(chunks)} chunks")

# Create vector store — LangChain + FAISS handles this in 1 line
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f"✅ Vector store created")

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print(f"✅ Retriever ready")

📄 Upload a PDF document...


Saving esg_policy.pdf to esg_policy.pdf
✅ Loaded: 1 pages
✅ Split into 6 chunks
✅ Vector store created
✅ Retriever ready


In [8]:
# Define the prompt template
prompt = ChatPromptTemplate.from_template("""
You are a document analysis expert for banking and compliance.
Answer the question based ONLY on the following context.
If the answer is not in the context, say "I cannot find this in the document."

Context:
{context}

Question: {question}

Answer:""")

# Build the chain — this is the magic of LangChain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain built")
print("Ready to answer questions!")

✅ RAG chain built
Ready to answer questions!


In [9]:
questions = [
    "What is the deadline for ESG reporting?",
    "Which sectors require DNSH assessment?",
    "What metrics must clients report?",
    "What happens if a client doesn't submit their report?"
]

print("🤖 LANGCHAIN RAG — Document Q&A")
print("=" * 55)

for question in questions:
    print(f"\n❓ {question}")
    answer = rag_chain.invoke(question)
    print(f"💬 {answer}")
    print("-" * 55)

🤖 LANGCHAIN RAG — Document Q&A

❓ What is the deadline for ESG reporting?
💬 The deadline for ESG reporting is 31 March each year.
-------------------------------------------------------

❓ Which sectors require DNSH assessment?
💬 High Risk sectors, including Energy, Manufacturing, and Transport, require DNSH assessment.
-------------------------------------------------------

❓ What metrics must clients report?
💬 According to Section 2 — Reporting Requirements, clients must report the following metrics: 
- GHG Scope 1 and Scope 2 emissions in tonnes CO2 equivalent
- Energy consumption in MWh.
-------------------------------------------------------

❓ What happens if a client doesn't submit their report?
💬 If a client doesn't submit their report, they face: 
1. Credit facility review
2. Potential loan covenant breach
3. Escalation to senior relationship management. Additionally, they will be flagged for relationship review.
-------------------------------------------------------
